#**Proyecto Integrador - Minería de Datos 1**

**02 - Calidad y Limpieza de Datos**


## Carga del dataset original

In [4]:
import pandas as pd
import numpy as np

df = pd.read_json('../data/raw/streaming_users_dirty.json')

#df = pd.read_json('/content/drive/MyDrive/streaming_users_dirty.json')

TOTAL_ORIGINAL = len(df)
print(f'Filas iniciales: {TOTAL_ORIGINAL}')
df.head()

Filas iniciales: 8160


,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
0,10000,39,Estándar,805.8,Brasil,Crime,2025-03-04,99
1,10001,37,Estándar,1173.4,Colombia,Crime,2019-04-02,2
2,10002,28,Básico,401.0,Colombia,Crime,2018-04-13,0
3,10003,43,Básico,62.4,Uruguay,Thriller,2021-01-31,0
4,10004,51,Básico,477.8,Perú,Thriller,2020-09-30,1


## Registro de auditoría (pipeline log)

La función `registrar()` guarda el estado del dataset después de cada
transformación.

Se registran las columnas: **Paso | Descripción | Filas | Nulos totales | Retención (%)**.

In [5]:
log = []

def registrar(paso, descripcion, dataframe):
    """Registra el estado actual del dataset en el log de auditoría."""
    filas = len(dataframe)
    nulos = dataframe.isnull().sum().sum()
    retencion = round(filas / TOTAL_ORIGINAL * 100, 2)
    log.append({
        'Paso': paso,
        'Descripcion': descripcion,
        'Filas': filas,
        'Nulos': nulos,
        'Retencion_%': retencion
    })
    print(f'[{paso}] {descripcion}')
    print(f'  Filas: {filas} | Nulos totales: {nulos} | Retención: {retencion}%\n')

registrar('00', 'Estado inicial del dataset', df)

[00] Estado inicial del dataset
  Filas: 8160 | Nulos totales: 753 | Retención: 100.0%



## Paso 1 — Eliminación de filas completamente duplicadas

* La inspección inicial detectó 126 filas completamente duplicadas
(todas las columnas idénticas).

* Una fila completamente duplicada representa la misma observación
cargada más de una vez en el sistema. No aporta información adicional y distorsiona cualquier
cálculo estadístico.                                                          
* La decisión estándar es eliminar los duplicados conservando la primera
ocurrencia.


In [6]:
print(f'Filas antes: {len(df)}')
print(f'Filas completamente duplicadas: {df.duplicated().sum()}')

df = df.drop_duplicates(keep='first')

print(f'Filas después: {len(df)}')
registrar('01', 'Eliminación de filas completamente duplicadas (keep=first)', df)

Filas antes: 8160
Filas completamente duplicadas: 126
Filas después: 8034
[01] Eliminación de filas completamente duplicadas (keep=first)
  Filas: 8034 | Nulos totales: 753 | Retención: 98.46%



## Paso 2 — Eliminación de duplicados por `user_id`

* Luego de eliminar filas exactas, aún hay registros con `user_id`
repetido pero con datos distintos en otras columnas.

* `user_id` es un identificador único: cada
valor debe corresponder a exactamente un usuario. Si un mismo `user_id` aparece con datos
distintos, no es posible determinar cuál versión es la correcta.

* La decisión conservadora es mantener la primera ocurrencia y descartar las siguientes.

In [7]:
print(f'Filas antes: {len(df)}')
print(f'user_id duplicados: {df["user_id"].duplicated().sum()}')

df = df.drop_duplicates(subset='user_id', keep='first')

print(f'Filas después: {len(df)}')
registrar('02', 'Eliminación de duplicados por user_id (keep=first)', df)

Filas antes: 8034
user_id duplicados: 34
Filas después: 8000
[02] Eliminación de duplicados por user_id (keep=first)
  Filas: 8000 | Nulos totales: 753 | Retención: 98.04%



## Paso 3 — Normalización de variables categóricas: limpieza de espacios
* Al inspeccionar los valores exactos de `subscription_plan` y
`country`, se detectaron variantes con espacios en blanco al inicio o al final del valor
(por ejemplo `'Premium '`, `'Estándar '`, `'Chile '`, `'Argentina '`). Estos espacios
son invisibles en una inspección visual pero hacen que Python trate esos valores como
categorías distintas.

* Utilizo el método `.str.strip()` para eliminar espacios en los extremos de cada
valor de texto. Se realiza como primer paso de normalización.


In [8]:
cols_str = ['subscription_plan', 'country', 'favorite_genre']
for col in cols_str:
    df[col] = df[col].str.strip()

print('Espacios eliminados en columnas:', cols_str)
registrar('03', 'Eliminación de espacios en columnas categóricas (str.strip)', df)

Espacios eliminados en columnas: ['subscription_plan', 'country', 'favorite_genre']
[03] Eliminación de espacios en columnas categóricas (str.strip)
  Filas: 8000 | Nulos totales: 753 | Retención: 98.04%



## Paso 4 — Normalización de `subscription_plan`

* La misma categoría aparece escrita de múltiples formas:
"Básico", "basico", "BASICO", "Basic"; "Estándar", "Std", "estandar", "STANDARD";
"Premium", "PREMIUM", "premium", "Premiun" (error tipográfico).

* Esta inconsistencia es un problema de *calidad de datos categóricos*:
* La normalización unifica todas
las variantes en una representación canónica.

* Se realiza el mapeo explícito de todas las variantes a tres categorías:
`Básico`, `Estándar`, `Premium`.

In [9]:
print('Valores únicos antes de normalizar:')
print(df['subscription_plan'].value_counts())

mapa_plan = {
    'Básico': 'Básico', 'basico': 'Básico', 'BASICO': 'Básico',
    'Basic': 'Básico', 'básico': 'Básico',
    'Estándar': 'Estándar', 'estandar': 'Estándar',
    'STANDARD': 'Estándar', 'Std': 'Estándar',
    'Premium': 'Premium', 'PREMIUM': 'Premium', 'premium': 'Premium',
    'Premiun': 'Premium'
}

df['subscription_plan'] = df['subscription_plan'].map(mapa_plan)

print('\nValores únicos después de normalizar:')
print(df['subscription_plan'].value_counts())
print(f'\nNulos generados (variantes no contempladas): {df["subscription_plan"].isnull().sum()}')
registrar('04', 'Normalización de subscription_plan a 3 categorías canónicas', df)

Valores únicos antes de normalizar:
subscription_plan
Básico      3386
Estándar    2699
Premium     1512
basico        60
BASICO        52
Basic         52
básico        50
Std           48
estandar      36
STANDARD      34
PREMIUM       26
Premiun       23
premium       22
Name: count, dtype: int64

Valores únicos después de normalizar:
subscription_plan
Básico      3600
Estándar    2817
Premium     1583
Name: count, dtype: int64

Nulos generados (variantes no contempladas): 0
[04] Normalización de subscription_plan a 3 categorías canónicas
  Filas: 8000 | Nulos totales: 753 | Retención: 98.04%



## Paso 5 — Normalización de `country`

* La variable `country` presenta el mismo país escrito de múltiples
formas. El dataset incluye
usuarios de 8 países: Argentina, Brasil, Chile, Colombia, México, Perú, Uruguay y Venezuela.

* Es el mismo problema de calidad categórica que `subscription_plan`.
* Para el análisis geográfico que deseo realizar más adelante, es imprescindible que cada país
quede representado por un único valor canónico.
* Se elige el nombre completo en español.

In [10]:
print('Valores únicos antes de normalizar:')
print(df['country'].value_counts())

mapa_pais = {
    'Argentina': 'Argentina', 'ARG': 'Argentina', 'argentina': 'Argentina',
    'Brasil': 'Brasil', 'Brazil': 'Brasil', 'BRA': 'Brasil', 'brasil': 'Brasil',
    'Chile': 'Chile', 'CHL': 'Chile', 'chile': 'Chile',
    'Colombia': 'Colombia', 'COL': 'Colombia', 'colombia': 'Colombia',
    'México': 'México', 'Mexico': 'México', 'MEX': 'México', 'méxico': 'México',
    'Perú': 'Perú', 'Peru': 'Perú', 'PER': 'Perú', 'perú': 'Perú',
    'Uruguay': 'Uruguay', 'URY': 'Uruguay', 'uruguay': 'Uruguay',
    'Venezuela': 'Venezuela', 'VEN': 'Venezuela', 'venezuela': 'Venezuela'
}

df['country'] = df['country'].map(mapa_pais)

print('\nValores únicos después de normalizar:')
print(df['country'].value_counts())
print(f'\nNulos generados (variantes no contempladas): {df["country"].isnull().sum()}')
registrar('05', 'Normalización de country a nombre completo en español (8 países)', df)

Valores únicos antes de normalizar:
country
Chile        1131
Brasil       1107
Uruguay      1102
México       1100
Colombia     1096
Perú         1091
Argentina    1079
colombia       27
méxico         25
uruguay        24
Brazil         21
COL            19
CHL            18
URY            17
MEX            16
argentina      16
PER            16
Mexico         15
chile          15
Peru           15
BRA            15
brasil         13
perú           12
ARG            10
Name: count, dtype: int64

Valores únicos después de normalizar:
country
Chile        1164
México       1156
Brasil       1156
Uruguay      1143
Colombia     1142
Perú         1134
Argentina    1105
Name: count, dtype: int64

Nulos generados (variantes no contempladas): 0
[05] Normalización de country a nombre completo en español (8 países)
  Filas: 8000 | Nulos totales: 753 | Retención: 98.04%



## Paso 6 — Normalización de `favorite_genre` e imputación de nulos

* La variable presenta inconsistencia de escritura (Crime/Crimen/CRIME,
"thriler" con error tipográfico, anglicismos como "Action", "Horror", "Comedy") y 240 nulos
(2.94% del dataset).

* Para determinar el mecanismo de falta, se compara la
tasa de nulos entre grupos de otras variables. El análisis no muestra variación sistemática
entre planes de suscripción, lo que descarta un mecanismo **MNAR** evidente.
* Asumiendo
**MCAR**, se aplica imputación con la **moda** (valor más frecuente), criterio apropiado
para variables categóricas.

In [11]:
print('Valores únicos antes de normalizar:')
print(df['favorite_genre'].value_counts(dropna=False))

mapa_genero = {
    'Acción': 'Acción', 'Accion': 'Acción', 'accion': 'Acción', 'Action': 'Acción',
    'Comedia': 'Comedia', 'comedia': 'Comedia', 'Comedy': 'Comedia',
    'Drama': 'Drama', 'drama': 'Drama',
    'Terror': 'Terror', 'terror': 'Terror', 'Horror': 'Terror',
    'Thriller': 'Thriller', 'thriller': 'Thriller', 'thriler': 'Thriller', 'THRILLER': 'Thriller',
    'Ciencia Ficción': 'Ciencia Ficción', 'Sci-Fi': 'Ciencia Ficción',
    'ciencia ficcion': 'Ciencia Ficción', 'SciFi': 'Ciencia Ficción',
    'Crimen': 'Crimen', 'Crime': 'Crimen', 'crimen': 'Crimen', 'CRIME': 'Crimen',
    'Documental': 'Documental', 'documental': 'Documental', 'Documentary': 'Documental',
    'Romance': 'Romance', 'romance': 'Romance',
    'Animación': 'Animación', 'Animation': 'Animación', 'animacion': 'Animación'
}

df['favorite_genre'] = df['favorite_genre'].map(mapa_genero)

print(f'Nulos tras normalización (incluye originales): {df["favorite_genre"].isnull().sum()}')

moda_genero = df['favorite_genre'].mode()[0]
print(f'Moda usada para imputación: {moda_genero}')

df['favorite_genre'] = df['favorite_genre'].fillna(moda_genero)

print('\nValores únicos después de normalizar e imputar:')
print(df['favorite_genre'].value_counts())
registrar('06', f'Normalización de favorite_genre e imputación con moda ({moda_genero})', df)

Valores únicos antes de normalizar:
favorite_genre
Comedia        1106
Drama          1089
Romance        1084
Thriller       1084
Documental     1066
Acción         1059
Crime          1044
None            240
Action           22
COMEDIA          19
Crimen           18
CRIME            17
Documentary      16
DRAMA            16
DOC              15
ACCIÓN           14
THRILLER         14
ROMANCE          14
comedy           12
romance          12
documental       10
drama            10
accion            8
thriler           6
crime             5
Name: count, dtype: int64
Nulos tras normalización (incluye originales): 335
Moda usada para imputación: Comedia

Valores únicos después de normalizar e imputar:
favorite_genre
Comedia       1441
Thriller      1104
Drama         1099
Romance       1096
Documental    1092
Acción        1089
Crimen        1079
Name: count, dtype: int64
[06] Normalización de favorite_genre e imputación con moda (Comedia)
  Filas: 8000 | Nulos totales: 513 | Retenci

## Paso 7 — Eliminación de valores imposibles en `age`

* `age` presenta 21 registros con edad < 0 y 53 con edad > 100
(mínimo = -5, máximo = 150).

* Estos valores no son outliers estadísticos dentro de una distribución
real — son errores de carga.
* Un *outlier estadístico* es un valor extremo pero admisible
dentro del dominio (por ejemplo, un usuario de 90 años); un *valor imposible* queda fuera
del dominio del problema por definición (edad negativa o 150 años).
* Para valores imposibles
la acción tomada es eliminar el registro, no imputar: imputar introduciría un dato ficticio
donde el original es incoherente. El rango razonable establecido es 0 ≤ age ≤ 100.



In [12]:
mask_invalidos = (df['age'] < 0) | (df['age'] > 100)
print(f'Registros con edad fuera de rango (0-100): {mask_invalidos.sum()}')
print(df[mask_invalidos][['user_id', 'age']].head(10))

df = df[~mask_invalidos].copy()

print(f'\nFilas después: {len(df)}')
print(f'Rango de age resultante: {df["age"].min()} - {df["age"].max()}')
registrar('07', 'Eliminación de registros con age < 0 o age > 100 (valores imposibles)', df)

Registros con edad fuera de rango (0-100): 74
      user_id  age
194     10194  130
324     10324  130
426     10426  150
442     10442   -5
529     10529  130
573     10573   -5
640     10640  150
655     10655  130
751     10751  150
1028    11028  150

Filas después: 7926
Rango de age resultante: 0 - 80
[07] Eliminación de registros con age < 0 o age > 100 (valores imposibles)
  Filas: 7926 | Nulos totales: 509 | Retención: 97.13%



## Paso 8 — Eliminación de valores imposibles en `monthly_watch_time_mins`

* La columna presenta 49 valores negativos (mínimo = -120) y
20 registros con el valor exacto 99999.

* *Valores negativos:* un tiempo de visualización no puede ser negativo por definición
  del dominio. Son errores de carga y se eliminan.
* *Valor centinela 99999:* patrón común en sistemas legados donde un código numérico fijo
  se usa para indicar "dato no disponible" en lugar de un nulo explícito.
* El valor 99999
  minutos equivale a más de 69 días continuos de visualización en un mes, lo cual es
  físicamente imposible. Se elimina porque indica una falla en el sistema de origen,
  no un dato real extremo.

In [13]:
mask_tiempo = (df['monthly_watch_time_mins'] < 0) | (df['monthly_watch_time_mins'] == 99999)
print(f'Registros con tiempo de visualización imposible: {mask_tiempo.sum()}')
print(f'  - Negativos: {(df["monthly_watch_time_mins"] < 0).sum()}')
print(f'  - Centinela (99999): {(df["monthly_watch_time_mins"] == 99999).sum()}')

df = df[~mask_tiempo].copy()

print(f'\nFilas después: {len(df)}')
registrar('08', 'Eliminación de valores negativos y centinela (99999) en monthly_watch_time_mins', df)

Registros con tiempo de visualización imposible: 69
  - Negativos: 49
  - Centinela (99999): 20

Filas después: 7857
[08] Eliminación de valores negativos y centinela (99999) en monthly_watch_time_mins
  Filas: 7857 | Nulos totales: 504 | Retención: 96.29%



## Paso 9 — Winsorización de outliers en `monthly_watch_time_mins`

* Luego de eliminar imposibles, el análisis IQR muestra:
  * Q1 = 492.7 min, Q3 = 1045.4 min, IQR = 552.6 min
  * Límite superior k = 1.5: 1874.3 min → 135 valores por encima
  * Límite superior k = 3.0: 2703.3 min → 119 valores por encima

* El criterio IQR con k = 1.5 identifica outliers *moderados*;
con k = 3.0, outliers *extremos*. A diferencia de los valores imposibles del paso
anterior, estos valores son altos pero razonables (un usuario "heavy user" podría
ver más de 31 horas al mes).
* Se aplica *winsorización* con k = 3.0: los valores
por encima del límite se reemplazan por ese límite, conservando el registro pero
reduciendo el impacto desproporcionado en los cálculos.
* Se elige k = 3.0 para ser
conservadores y no eliminar comportamiento real del dataset.

In [ ]:
Q1 = df['monthly_watch_time_mins'].quantile(0.25)
Q3 = df['monthly_watch_time_mins'].quantile(0.75)
IQR = Q3 - Q1
lim_sup = Q3 + 3.0 * IQR

print(f'Q1 = {Q1:.1f} | Q3 = {Q3:.1f} | IQR = {IQR:.1f}')
print(f'Límite superior (k=3.0): {lim_sup:.1f}')
print(f'Valores winsorizados: {(df["monthly_watch_time_mins"] > lim_sup).sum()}')

df['monthly_watch_time_mins'] = df['monthly_watch_time_mins'].clip(upper=lim_sup)

print(f'Máximo después de winsorizar: {df["monthly_watch_time_mins"].max():.1f}')
registrar('09', f'Winsorización de monthly_watch_time_mins (límite superior IQR k=3.0: {lim_sup:.1f})', df)

## Paso 10 — Imputación de nulos en `monthly_watch_time_mins`

* Quedan 193 valores nulos en `monthly_watch_time_mins`.
* Antes de imputar se diagnostica el *mecanismo de falta*:
  - **MCAR** (Missing Completely At Random): la probabilidad de que falte no depende de
  ninguna otra variable → imputar con mediana/media global.
  - **MAR** (Missing At Random): la probabilidad depende de otras variables observadas.
  - **MNAR** (Missing Not At Random): el faltante está relacionado con el propio valor ausente.

* No se observa variación sistemática de la tasa de nulos entre planes o países, lo que
descarta MAR/MNAR evidente. Asumiendo **MCAR**, se imputa con la **mediana**, más robusta
que la media ante la asimetría de la distribución de tiempos de visualización.

In [14]:
print(f'Nulos antes: {df["monthly_watch_time_mins"].isnull().sum()}')

mediana = df['monthly_watch_time_mins'].median()
print(f'Mediana usada para imputación: {mediana:.2f} min')

df['monthly_watch_time_mins'] = df['monthly_watch_time_mins'].fillna(mediana)

print(f'Nulos después: {df["monthly_watch_time_mins"].isnull().sum()}')
registrar('10', f'Imputación de nulos en monthly_watch_time_mins con mediana ({mediana:.2f})', df)

Nulos antes: 192
Mediana usada para imputación: 758.50 min
Nulos después: 0
[10] Imputación de nulos en monthly_watch_time_mins con mediana (758.50)
  Filas: 7857 | Nulos totales: 312 | Retención: 96.29%



## Paso 11 — Eliminación de valores imposibles en `customer_support_tickets`

* Hay 29 registros con valor negativo (mínimo = -1).

* La cantidad de tickets de soporte es una variable de conteo por lo tanto no puede ser negativa. Son errores de carga (valores imposibles),
no outliers de una distribución real.
* Se eliminan.

In [15]:
print(f'Registros con tickets negativos: {(df["customer_support_tickets"] < 0).sum()}')

df = df[df['customer_support_tickets'] >= 0].copy()

print(f'Filas después: {len(df)}')
registrar('11', 'Eliminación de registros con customer_support_tickets < 0', df)

Registros con tickets negativos: 27
Filas después: 7830
[11] Eliminación de registros con customer_support_tickets < 0
  Filas: 7830 | Nulos totales: 310 | Retención: 95.96%



## Paso 12 — Winsorización de outliers en `customer_support_tickets`

* Luego de eliminar negativos:
  - Q1 = 0, Q3 = 1, IQR = 1
  - Límite superior k = 3.0: 4.0 → 81 valores por encima
  - Máximo observado: 150

* El IQR muy pequeño (= 1) refleja que la mayoría de los usuarios
tiene 0 o 1 tickets.
* Los valores superiores a 4 (k=3.0) son extremadamente inusuales.
* El máximo de 150 tickets es altamente sospechoso, podría ser un usuario de prueba o
un error de sistema.
* Se aplica *winsorización con k = 3.0* para conservar los registros
pero limitar el efecto de esos valores extremos en el análisis de correlación con `age`.


In [16]:
Q1 = df['customer_support_tickets'].quantile(0.25)
Q3 = df['customer_support_tickets'].quantile(0.75)
IQR = Q3 - Q1
lim_sup = Q3 + 3.0 * IQR

print(f'Q1 = {Q1} | Q3 = {Q3} | IQR = {IQR}')
print(f'Límite superior (k=3.0): {lim_sup}')
print(f'Valores winsorizados: {(df["customer_support_tickets"] > lim_sup).sum()}')

df['customer_support_tickets'] = df['customer_support_tickets'].clip(upper=lim_sup)

print(f'Máximo después de winsorizar: {df["customer_support_tickets"].max()}')
registrar('12', f'Winsorización de customer_support_tickets (límite superior IQR k=3.0: {lim_sup})', df)

Q1 = 0.0 | Q3 = 1.0 | IQR = 1.0
Límite superior (k=3.0): 4.0
Valores winsorizados: 81
Máximo después de winsorizar: 4
[12] Winsorización de customer_support_tickets (límite superior IQR k=3.0: 4.0)
  Filas: 7830 | Nulos totales: 310 | Retención: 95.96%



## Paso 13 — Limpieza de `last_login_date`

* La columna presenta tres problemas:
  - 320 valores nulos originales
  - 449 valores no parseables (formato de fecha roto)
  - 15 fechas posteriores al 29/06/2026 (fecha de referencia del proyecto)

* Los *no parseables* son errores de formato del sistema de origen. Con `errors='coerce'`
  quedan como `NaT` (equivalente al nulo para fechas en pandas).
* Las *fechas futuras* son valores imposibles para un campo de "último login" — un
  usuario no puede haber iniciado sesión en el futuro. Se tratan como valores imposibles:
  eliminación del registro.
* Los *nulos* resultantes se eliminan porque `last_login_date` no es variable central
  en las preguntas de análisis.

In [17]:
df['last_login_date'] = pd.to_datetime(df['last_login_date'], errors='coerce')

print(f'Nulos tras conversión (originales + no parseables): {df["last_login_date"].isnull().sum()}')

fecha_corte = pd.Timestamp('2026-06-29')
futuras = (df['last_login_date'] > fecha_corte).sum()
print(f'Fechas futuras (> {fecha_corte.date()}): {futuras}')
df = df[~(df['last_login_date'] > fecha_corte)].copy()

nulos_fecha = df['last_login_date'].isnull().sum()
print(f'Registros con last_login_date nulo: {nulos_fecha}')
df = df.dropna(subset=['last_login_date']).copy()

print(f'\nFilas después: {len(df)}')
print(f'Rango de fechas: {df["last_login_date"].min().date()} → {df["last_login_date"].max().date()}')
registrar('13', 'Conversión de last_login_date + eliminación de fechas futuras y nulos', df)

Nulos tras conversión (originales + no parseables): 751
Fechas futuras (> 2026-06-29): 15
Registros con last_login_date nulo: 751

Filas después: 7064
Rango de fechas: 2018-01-01 → 2025-12-31
[13] Conversión de last_login_date + eliminación de fechas futuras y nulos
  Filas: 7064 | Nulos totales: 0 | Retención: 86.57%



## Verificación del dataset limpio



In [18]:
print('=== Estado final del dataset ===')
print(f'Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas')
print(f'Retención: {len(df)} de {TOTAL_ORIGINAL} filas ({len(df)/TOTAL_ORIGINAL*100:.1f}%)')
print()
print('=== Nulos por columna ===')
print(df.isnull().sum())
print()
print('=== Tipos de datos ===')
print(df.dtypes)
print()
print('=== Estadísticas descriptivas (numéricas) ===')
print(df[['age', 'monthly_watch_time_mins', 'customer_support_tickets']].describe().round(2))
print()
print('=== Variables categóricas ===')
for col in ['subscription_plan', 'country', 'favorite_genre']:
    vals = sorted(df[col].dropna().unique())
    print(f'{col}: {vals}')

=== Estado final del dataset ===
Dimensiones: 7064 filas x 8 columnas
Retención: 7064 de 8160 filas (86.6%)

=== Nulos por columna ===
user_id                     0
age                         0
subscription_plan           0
monthly_watch_time_mins     0
country                     0
favorite_genre              0
last_login_date             0
customer_support_tickets    0
dtype: int64

=== Tipos de datos ===
user_id                              int64
age                                  int64
subscription_plan                   object
monthly_watch_time_mins            float64
country                             object
favorite_genre                      object
last_login_date             datetime64[ns]
customer_support_tickets             int64
dtype: object

=== Estadísticas descriptivas (numéricas) ===
           age  monthly_watch_time_mins  customer_support_tickets
count  7064.00                  7064.00                   7064.00
mean     33.46                   865.17            

In [ ]:
df.to_csv('../data/processed/streaming_users_clean.csv', index=False)
print('Dataset procesado guardado en: data/processed/streaming_users_clean.csv')
print(f'Dimensiones finales: {df.shape}')

In [ ]:
log_df = pd.DataFrame(log)
log_df.to_csv('../logs/pipeline_log.csv', index=False)

print('Log de auditoría guardado en: logs/pipeline_log.csv')
print()
print(log_df.to_string(index=False))